<a href="https://colab.research.google.com/github/bananmagdy3-crypto/Olist-dataset/blob/main/Olist_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import kagglehub
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


In [5]:
import pandas as pd

orders = pd.read_csv(f"{path}/olist_orders_dataset.csv")
items = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
products = pd.read_csv(f"{path}/olist_products_dataset.csv")
customers = pd.read_csv(f"{path}/olist_customers_dataset.csv")
sellers = pd.read_csv(f"{path}/olist_sellers_dataset.csv")

print(orders.shape)

(99441, 8)


In [6]:
orders = orders[orders['order_status'] == 'delivered'].copy()
df = orders.merge(items, on='order_id', how='inner')
df = df.merge(products, on='product_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')

print(df.shape)

(110197, 29)


In [7]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_length_cm,product_height_cm,product_width_cm,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,19.0,8.0,13.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,9350,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,19.0,13.0,19.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,31570,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,24.0,19.0,21.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,14840,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,...,30.0,10.0,20.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,31842,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,...,51.0,15.0,15.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,8752,mogi das cruzes,SP


In [8]:
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

df = df.dropna(subset=['order_delivered_customer_date']).copy()

df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)

df = df.drop(columns=['order_delivered_customer_date', 'order_delivered_carrier_date'], errors='ignore')

print(df['is_late'].value_counts(normalize=True))

is_late
0    0.920918
1    0.079082
Name: proportion, dtype: float64


In [9]:
import numpy as np

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

df['approval_hours'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 3600


df['estimated_delivery_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / (3600 * 24)

df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

df['freight_to_price_ratio'] = df['freight_value'] / (df['price'] + 1e-5)
df['product_volume_cm3'] = (
    df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
)

df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_month'] = df['order_purchase_timestamp'].dt.month


date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_estimated_delivery_date', 'shipping_limit_date'
]
df = df.drop(columns=date_cols, errors='ignore')

In [10]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Select features (excluding IDs, raw timestamps, and target)
num_cols = [
    "approval_hours",
    "estimated_delivery_days",
    "same_state",
    "freight_to_price_ratio",
    "product_volume_cm3",
    "purchase_dayofweek",
    "purchase_month",
    "price",
    "freight_value",
    "product_weight_g",
]

cat_cols = [
    "customer_state",
    "seller_state",
    "product_category_name",
]

# Keep only columns that exist in df
num_cols = [c for c in num_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

X = df[num_cols + cat_cols].copy()
y = df["is_late"].copy()

# Optional: drop rows where target itself is missing (safety check)
valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx].astype(int)

# 2. Stratified Train / Test split (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train late rate:", y_train.mean().round(4))
print("Test late rate: ", y_test.mean().round(4))

# 3. Preprocessing + Logistic Regression baseline pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
        ),
    ]
)

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)[:, 1]

print("\n--- Baseline Logistic Regression Results ---")
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba), 4))
print("F1 (Late):", round(f1_score(y_test, y_pred, pos_label=1), 4))
print("Recall (Late):", round(recall_score(y_test, y_pred, pos_label=1), 4))
print("Precision (Late):", round(precision_score(y_test, y_pred, pos_label=1), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

Train late rate: 0.0791
Test late rate:  0.0791

--- Baseline Logistic Regression Results ---
ROC-AUC:   0.7109
F1 (Late): 0.2289
Recall (Late): 0.6472
Precision (Late): 0.1391

Classification Report:
               precision    recall  f1-score   support

           0     0.9558    0.6559    0.7779     20295
           1     0.1391    0.6472    0.2289      1743

    accuracy                         0.6552     22038
   macro avg     0.5474    0.6515    0.5034     22038
weighted avg     0.8912    0.6552    0.7345     22038



In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# HistGradientBoosting يتعامل مع المصفوفة الكثيفة بسهولة
# نستخدم OneHotEncoder(sparse_output=False) في preprocessor خاص بالموديل الشجري
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            cat_cols,
        ),
    ]
)

hgb_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(
                class_weight="balanced",
                max_iter=200,
                learning_rate=0.05,
                random_state=42,
            ),
        ),
    ]
)

hgb_model.fit(X_train, y_train)

y_pred_tree = hgb_model.predict(X_test)
y_proba_tree = hgb_model.predict_proba(X_test)[:, 1]

print("--- HistGradientBoosting Results ---")
print("ROC-AUC:         ", round(roc_auc_score(y_test, y_proba_tree), 4))
print("F1 (Late):       ", round(f1_score(y_test, y_pred_tree, pos_label=1), 4))
print("Recall (Late):   ", round(recall_score(y_test, y_pred_tree, pos_label=1), 4))
print("Precision (Late):", round(precision_score(y_test, y_pred_tree, pos_label=1), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tree, digits=4))

--- HistGradientBoosting Results ---
ROC-AUC:          0.8034
F1 (Late):        0.32
Recall (Late):    0.6827
Precision (Late): 0.209

Classification Report:
               precision    recall  f1-score   support

           0     0.9662    0.7781    0.8620     20295
           1     0.2090    0.6827    0.3200      1743

    accuracy                         0.7705     22038
   macro avg     0.5876    0.7304    0.5910     22038
weighted avg     0.9063    0.7705    0.8191     22038



In [12]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, precision_score, recall_score

# 1. Permutation Importance on the test set
# Note: passing the full pipeline evaluates importance on raw X_test features cleanly
perm_result = permutation_importance(
    hgb_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importance_df = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    }
).sort_values("importance_mean", ascending=False)

print("--- Top Features by Permutation Importance (ROC-AUC drop) ---")
print(importance_df.to_string(index=False))

# 2. Threshold Tuning across candidate cutoffs
thresholds = np.arange(0.25, 0.85, 0.05)
rows = []

for thresh in thresholds:
    preds_t = (y_proba_tree >= thresh).astype(int)
    rows.append(
        {
            "threshold": round(float(thresh), 2),
            "precision_late": round(float(precision_score(y_test, preds_t, pos_label=1, zero_division=0)), 4),
            "recall_late": round(float(recall_score(y_test, preds_t, pos_label=1, zero_division=0)), 4),
            "f1_late": round(float(f1_score(y_test, preds_t, pos_label=1, zero_division=0)), 4),
            "flagged_late_pct": round(float(preds_t.mean()), 4),
        }
    )

thresh_df = pd.DataFrame(rows)
print("\n--- Threshold Trade-off Table ---")
print(thresh_df.to_string(index=False))

best_row = thresh_df.loc[thresh_df["f1_late"].idxmax()]
print(
    f"\nBest F1 threshold in grid: {best_row['threshold']} "
    f"(F1={best_row['f1_late']}, Precision={best_row['precision_late']}, Recall={best_row['recall_late']})"
)

--- Top Features by Permutation Importance (ROC-AUC drop) ---
                feature  importance_mean  importance_std
         purchase_month         0.137524        0.001974
estimated_delivery_days         0.117886        0.004008
         customer_state         0.072098        0.001028
          freight_value         0.027154        0.001735
             same_state         0.014684        0.001475
         approval_hours         0.012643        0.000898
           seller_state         0.011767        0.000621
  product_category_name         0.005059        0.000549
       product_weight_g         0.004853        0.000921
                  price         0.004511        0.000503
     product_volume_cm3         0.003876        0.000646
     purchase_dayofweek         0.002289        0.000215
 freight_to_price_ratio         0.001637        0.000322

--- Threshold Trade-off Table ---
 threshold  precision_late  recall_late  f1_late  flagged_late_pct
      0.25          0.1125       0.931

In [13]:
import joblib

joblib.dump(hgb_model, "olist_delay_hgb_model.joblib")

['olist_delay_hgb_model.joblib']